# Phase 3: Text Preprocessing & Data Quality Pipeline

## Objectives:
1. **Baseline Assessment**: Compare `Resume_str` (pre-parsed) vs `Resume_html` (original) parsing quality.
2. **Noise Removal**: Define and execute removal of URLs, emails, special chars, and stopwords.
3. **Normalization**: Apply lemmatization to reduce feature dimensionality.
4. **Outlier Filtering**: Remove resumes under 50 words (identified in Phase 2 EDA).
5. **Multi-Source Job Processing**: Clean `Job Description`, `skills`, and `Qualifications` columns.

In [6]:
import pandas as pd
import numpy as np
import sys
import os
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
from collections import Counter

# Local Imports
sys.path.append("../")
from src.data.preprocessor import preprocess_pipeline, clean_html

%matplotlib inline
sns.set_style("whitegrid")

## 1. What is being removed?
To build a high-signal vector space, we remove the following:
- **HTML Markup**: Tags like `<div>` or `<b>` which add no semantic value.
- **Web Metadata**: URLs and Email addresses (confidentiality & noise).
- **Punctuation & Special Characters**: Symbols that don't contribute to skill identification.
- **Stopwords**: High-frequency words (the, is, and) that drown out technical signals.
- **Whitespace**: Normalizing to single spaces for consistent tokenization.

## 2. Load and Compare Raw Sources
We compare the provided `Resume_str` with our own HTML parsing to ensure no data was lost in the original source.

In [8]:
df_resumes = pd.read_csv("../data/raw/resumes/Resume/Resume.csv")

# Compare first 3 samples
for i in range(3):
    print(f"--- Sample {i} ---")
    print(f"Original Str (len): {len(str(df_resumes.iloc[i]['Resume_str']))}")
    print(f"Recleaned HTML (len): {len(clean_html(str(df_resumes.iloc[i]['Resume_html'])))}")
    print("-" * 20)

--- Sample 0 ---
Original Str (len): 5442
Recleaned HTML (len): 5935
--------------------
--- Sample 1 ---
Original Str (len): 5572
Recleaned HTML (len): 6008
--------------------
--- Sample 2 ---
Original Str (len): 7720
Recleaned HTML (len): 8165
--------------------


## 3. Preprocessing Jobs: Multi-Column Focus
The roadmap requires cleaning `Job Description`, `skills`, and `Qualifications` to merge them into a single context.

In [9]:
df_jobs = pd.read_csv("../data/raw/jobs/job_descriptions.csv", nrows=10000)

target_cols = ['Job Description', 'skills', 'Qualifications']

for col in target_cols:
    tqdm.pandas(desc=f"Cleaning {col}")
    df_jobs[f'cleaned_{col.lower().replace(" ", "_")}'] = df_jobs[col].progress_apply(preprocess_pipeline)

# Create a combined feature column
df_jobs['combined_context'] = df_jobs[['cleaned_job_description', 'cleaned_skills', 'cleaned_qualifications']].fillna('').agg(' '.join, axis=1)
df_jobs.head(2)

Cleaning Job Description:   0%|          | 0/10000 [00:00<?, ?it/s]

Cleaning skills:   0%|          | 0/10000 [00:00<?, ?it/s]

Cleaning Qualifications:   0%|          | 0/10000 [00:00<?, ?it/s]

,Job Id,Experience,Qualifications,Salary Range,location,Country,latitude,longitude,Work Type,Company Size,...,Job Description,Benefits,skills,Responsibilities,Company,Company Profile,cleaned_job_description,cleaned_skills,cleaned_qualifications,combined_context
0,1089843540111562,5 to 15 Years,M.Tech,$59K-$99K,Douglas,Isle of Man,54.2361,-4.5481,Intern,26801,...,Social Media Managers oversee an organizations...,"{'Flexible Spending Accounts (FSAs), Relocatio...","Social media platforms (e.g., Facebook, Twitte...","Manage and grow social media accounts, create ...",Icahn Enterprises,"{""Sector"":""Diversified"",""Industry"":""Diversifie...",social medium manager oversee organization soc...,social medium platform e g facebook twitter in...,tech,social medium manager oversee organization soc...
1,398454096642776,2 to 12 Years,BCA,$56K-$116K,Ashgabat,Turkmenistan,38.9697,59.5563,Intern,100340,...,Frontend Web Developers design and implement u...,"{'Health Insurance, Retirement Plans, Paid Tim...","HTML, CSS, JavaScript Frontend frameworks (e.g...","Design and code user interfaces for websites, ...",PNC Financial Services Group,"{""Sector"":""Financial Services"",""Industry"":""Com...",frontend web developer design implement user i...,html cs javascript frontend framework e g reac...,bca,frontend web developer design implement user i...


## 4. Cleaning Resumes & Filtering Outliers
Applying the pipeline and removing entries that are too short to be meaningful.

In [10]:
tqdm.pandas(desc="Cleaning Resumes")
df_resumes['cleaned_text'] = df_resumes['Resume_str'].progress_apply(preprocess_pipeline)

# Filter out resumes under 50 words
df_resumes['word_count'] = df_resumes['cleaned_text'].apply(lambda x: len(x.split()))
df_filtered = df_resumes[df_resumes['word_count'] >= 50].copy()

print(f"Original count: {len(df_resumes)}")
print(f"Filtered count: {len(df_filtered)}")
print(f"Dropped {len(df_resumes)-len(df_filtered)} low-quality resumes.")

Cleaning Resumes:   0%|          | 0/2484 [00:00<?, ?it/s]

Original count: 2484
Filtered count: 2483
Dropped 1 low-quality resumes.


## 5. Statistical Verification
Checking vocabulary reduction to ensure noise was minimized.

In [11]:
all_raw = " ".join(df_resumes['Resume_str'].astype(str)).split()
all_clean = " ".join(df_filtered['cleaned_text']).split()

print(f"Global Vocab Size (Raw): {len(set(all_raw))}")
print(f"Global Vocab Size (Cleaned): {len(set(all_clean))}")
print(f"Total Reduction: {((len(set(all_raw)) - len(set(all_clean)))/len(set(all_raw)))*100:.1f}%")

Global Vocab Size (Raw): 111423
Global Vocab Size (Cleaned): 36136
Total Reduction: 67.6%


## 6. Persistence
Saving the enhanced processed datasets.

In [12]:
os.makedirs("../data/processed", exist_ok=True)

df_filtered[['ID', 'Category', 'cleaned_text']].to_csv("../data/processed/resumes_cleaned.csv", index=False)
df_jobs[['Job Title', 'combined_context']].to_csv("../data/processed/jobs_cleaned.csv", index=False)

print("Processed data saved to data/processed/")

Processed data saved to data/processed/
